# UPI Transaction Fraudulent Predictive Modeling


## 1. Import Libraries

In [24]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

## 2. Connect to SQL Server

In [27]:
from sqlalchemy import create_engine
import urllib

server = r'ARYAS\SQLEXPRESS'
database = 'UPI_txn'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

connection_string = f"mssql+pyodbc:///?odbc_connect={params}"

engine = create_engine(connection_string)

conn = engine.connect()

print("✅ Connected Successfully")

✅ Connected Successfully


## 3. Load Data

In [30]:
# Load Data
transactions= pd.read_sql('select * from transactions', engine)
users= pd.read_sql('select * from users', engine)
merchant= pd.read_sql('select * from merchants', engine)
fraud= pd.read_sql('select * from fraud_labels', engine)

## 4. Merge Data

In [32]:
# Merge Tables
df= transactions.merge(users, how='left', on='user_id')
df= df.merge(fraud, how='left', on='transaction_id')
df= df.merge(merchant, left_on='receiver_id_x', right_on='merchant_id',  how='left')

## 5. Data Understanding

In [36]:
df.head()

,transaction_id,user_id_x,receiver_id_x,receiver_type,amount_x,timestamp_x,date,hour_of_day,day_of_week,is_weekend,...,amount_deviation_score_y,merchant_id,merchant_name,merchant_category,merchant_size,city_y,city_tier_y,avg_daily_transactions,is_registered,rating
0,TXN0000001,USR01603,MRC0374,Merchant,1650.839966,2024-12-26 14:52:00,2024-12-26,14,Thursday,False,...,0.5146,MRC0374,Merchant_374,Recharge,Medium,Hyderabad,Tier 1,37.0,False,2.8
1,TXN0000002,USR01759,MRC0274,Merchant,599.200012,2024-06-29 00:09:24,2024-06-29,0,Saturday,True,...,0.7026,MRC0274,Merchant_274,Insurance,Medium,Tiruchirappalli,Tier 3,71.0,True,4.3
2,TXN0000003,USR01894,MRC0122,Merchant,261.359985,2024-02-21 05:08:33,2024-02-21,5,Wednesday,False,...,0.1547,MRC0122,Merchant_122,Healthcare,Enterprise,Solapur,Tier 3,288.0,True,4.7
3,TXN0000004,USR00606,USR00692,User,215.710007,2024-08-09 18:52:33,2024-08-09,18,Friday,False,...,0.0078,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TXN0000005,USR01246,MRC0007,Merchant,411.149994,2024-02-18 06:37:11,2024-02-18,6,Sunday,True,...,0.1170,MRC0007,Merchant_7,Food & Dining,Medium,Ranchi,Tier 3,63.0,True,4.7


In [38]:
df.shape

(20000, 61)

In [40]:
df.isnull().sum().sum()

43929

In [42]:
nulls=df.isnull().sum().sort_values(ascending=False)
print(nulls.to_string())

rating                         4881
city_y                         4881
merchant_name                  4881
merchant_category              4881
merchant_size                  4881
merchant_id                    4881
city_tier_y                    4881
avg_daily_transactions         4881
is_registered                  4881
user_id_y                         0
kyc_status                        0
account_age_days                  0
linked_bank_count                 0
avg_monthly_transactions          0
avg_transaction_value             0
preferred_app                     0
preferred_device                  0
user_loyalty_score_y              0
is_high_risk_user                 0
is_fraud_y                        0
receiver_id_y                     0
amount_y                          0
timestamp_y                       0
city_x                            0
new_device_flag_y                 0
ip_location_mismatch_y            0
failed_attempts_last_24h_y        0
transaction_velocity_y      

## 6. Data Cleaning

In [45]:
# Handling Null Values
cat_cols= df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col]=df[col].fillna('unknown')

In [47]:
# Handling Null Values--> Numerical Columns
df['avg_daily_transactions']=df['avg_daily_transactions'].fillna(df['avg_daily_transactions'].median())
df['is_registered']=df['is_registered'].fillna(0)
df['rating']=df['rating'].fillna(df['rating'].median())

In [49]:
df.isnull().sum().sum()

0

In [51]:
# Check Data Types
df_data_types=df.dtypes
print(df_data_types.to_string())

transaction_id                         object
user_id_x                              object
receiver_id_x                          object
receiver_type                          object
amount_x                              float64
timestamp_x                    datetime64[ns]
date                                   object
hour_of_day                             int64
day_of_week                            object
is_weekend                               bool
is_night_transaction                     bool
time_since_last_txn_min               float64
transaction_type                       object
payment_app                            object
device_type                            object
status                                 object
user_city_tier                         object
user_kyc_status                        object
user_avg_monthly_txn                    int64
user_avg_txn_value                    float64
user_loyalty_score_x                  float64
new_device_flag_x                 

In [53]:
# Drop unnecessary duplicate columns

drop_cols = [
    'user_id_y',
    'receiver_id_y',
    'amount_y',
    'timestamp_y',
    'new_device_flag_y',
    'ip_location_mismatch_y',
    'failed_attempts_last_24h_y',
    'transaction_velocity_y',
    'amount_deviation_score_y',
    'is_fraud_y']

df = df.drop(columns=drop_cols, errors='ignore')

In [55]:
df = df.rename(columns={
    'user_id_x': 'USER_ID',
    'receiver_id_x': 'RECEIVER_ID',
    'amount_x': 'AMOUNT',
    'timestamp_x': 'TIMESTAMP',
    
    'new_device_flag_x': 'NEW_DEVICE_FLAG',
    'ip_location_mismatch_x': 'IP_LOCATION_MISMATCH',
    'failed_attempts_last_24h_x': 'FAILED_ATTEMPTS_LAST_24H',
    'transaction_velocity_x': 'TRANSACTION_VELOCITY',
    'amount_deviation_score_x': 'AMOUNT_DEVIATION_SCORE',
    'is_fraud_x': 'IS_FRAUD',
    
    'city_x': 'CITY',
    'city_tier_x': 'CITY_TIER'
})

In [57]:
df.shape

(20000, 51)

In [59]:
final_df=df.copy()

In [61]:
final_df= final_df.drop(columns= ['TRANSACTION_ID',
    'USER_ID',
    'RECEIVER_ID',
    'MERCHANT_ID',
    'TIMESTAMP',
    'DATE'], errors='ignore')

In [63]:
final_df.shape

(20000, 48)

## 7. Encoding

In [66]:
# Apply Label Encoding
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
cat_cols=final_df.select_dtypes(include=['object','bool']).columns

for col in cat_cols:
    final_df[col]=le.fit_transform(final_df[col].astype(str))
final_df.dtypes

transaction_id                   int32
receiver_type                    int32
AMOUNT                         float64
date                             int32
hour_of_day                      int64
day_of_week                      int32
is_weekend                       int32
is_night_transaction             int32
time_since_last_txn_min        float64
transaction_type                 int32
payment_app                      int32
device_type                      int32
status                           int32
user_city_tier                   int32
user_kyc_status                  int32
user_avg_monthly_txn             int64
user_avg_txn_value             float64
user_loyalty_score_x           float64
NEW_DEVICE_FLAG                  int32
IP_LOCATION_MISMATCH             int32
FAILED_ATTEMPTS_LAST_24H         int64
TRANSACTION_VELOCITY           float64
AMOUNT_DEVIATION_SCORE         float64
IS_FRAUD                         int32
recurring_payment_flag           int32
balance_after_transaction

In [67]:
final_df.columns= final_df.columns.str.lower()

## 8. Features & Target Split

In [71]:
# Feature & Target Split
x= final_df.drop('is_fraud', axis=1)
y= final_df[['is_fraud']]

In [73]:
x.head(2)

,transaction_id,receiver_type,amount,date,hour_of_day,day_of_week,is_weekend,is_night_transaction,time_since_last_txn_min,transaction_type,...,is_high_risk_user,merchant_id,merchant_name,merchant_category,merchant_size,city_y,city_tier_y,avg_daily_transactions,is_registered,rating
0,0,0,1650.839966,360,14,4,0,0,0.0,2,...,0,373,305,7,1,12,0,37.0,0,2.8
1,1,0,599.200012,180,0,2,1,1,0.0,5,...,0,273,194,6,1,33,2,71.0,1,4.3


In [75]:
y.head()

,is_fraud
0,0
1,0
2,0
3,0
4,0


## 9.Train Test Split

In [78]:
# Train Test Split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((16000, 47), (4000, 47), (16000, 1), (4000, 1))

## 10. Feature Selection

In [81]:
# Recursive feature elimination
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

In [83]:
model= LogisticRegression()
rfe=RFE(estimator= model, n_features_to_select=15)
rfe.fit(x_train,y_train)

RFE(estimator=LogisticRegression(), n_features_to_select=15)

In [84]:
rfe_ranking=pd.DataFrame({'Features': x_train.columns, 'Rank':rfe.ranking_})
rfe_ranking.sort_values(by='Rank')

,Features,Rank
23,recurring_payment_flag,1
41,merchant_size,1
36,user_loyalty_score_y,1
35,preferred_device,1
34,preferred_app,1
29,kyc_status,1
28,city_tier,1
45,is_registered,1
21,transaction_velocity,1
20,failed_attempts_last_24h,1


In [85]:
selected_features=x_train.columns[rfe.support_].tolist()
print(selected_features)

['user_city_tier', 'user_kyc_status', 'user_loyalty_score_x', 'new_device_flag', 'failed_attempts_last_24h', 'transaction_velocity', 'recurring_payment_flag', 'city_tier', 'kyc_status', 'preferred_app', 'preferred_device', 'user_loyalty_score_y', 'merchant_size', 'is_registered', 'rating']


In [86]:
x_train_sf=x_train[selected_features]
x_test_sf=x_test[selected_features]

In [91]:
from imblearn.over_sampling import SMOTE

In [93]:
smote=SMOTE(sampling_strategy=0.5 ,random_state=42)

In [95]:
x_train_smote, y_train_smote=smote.fit_resample(x_train_sf,y_train)
y_train_smote.value_counts()

is_fraud
0           15390
1            7695
Name: count, dtype: int64

In [189]:
# Import Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
rf=RandomForestClassifier(n_estimators=100, class_weight='balanced',random_state=42,max_depth=10, min_samples_split=120)
lr=LogisticRegression( class_weight='balanced',random_state=42)
dt=DecisionTreeClassifier(class_weight='balanced',random_state=42, max_depth=10, min_samples_split=20)

In [169]:
# RandomForestClassifier Model
rf.fit(x_train_smote,y_train_smote)
rf_pred=rf.predict(x_test_sf)
rf_pred[:5]

array([1, 0, 1, 0, 0])

In [100]:
# Logistic Regression Model
lr.fit(x_train_smote,y_train_smote)
lr_pred=lr.predict(x_test_sf)
lr_pred[:5]

array([1, 0, 1, 0, 0])

In [191]:
# Desiction Tree Model
dt.fit(x_train_smote,y_train_smote)
dt_pred=dt.predict(x_test_sf)
dt_pred[:5]

array([1, 0, 0, 0, 0])

In [102]:
# Import Metrics
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

In [171]:
# Accuracy Score
print('RandomForestClassifier Accuracy Score: ',accuracy_score(y_test,rf_pred))
print('LogisticRegression Accuracy Score: ',accuracy_score(y_test,lr_pred))
print('DecisionTreeClassifier Accuracy Score: ',accuracy_score(y_test,dt_pred))

RandomForestClassifier Accuracy Score:  0.754
LogisticRegression Accuracy Score:  0.65175
DecisionTreeClassifier Accuracy Score:  0.7595


In [173]:
# ROC AUC Score
print('RandomForestClassifier ROC AUC Score: ',roc_auc_score(y_test,rf_pred))
print('LogisticRegression ROC AUC Score: ',roc_auc_score(y_test,lr_pred))
print('DecisionTreeClassifier ROC AUC Score: ',roc_auc_score(y_test,dt_pred))

RandomForestClassifier ROC AUC Score:  0.5269278667189949
LogisticRegression ROC AUC Score:  0.602427661992793
DecisionTreeClassifier ROC AUC Score:  0.532925240107307


In [175]:
# Confusion Matrix
print('RandomForestClassifier Confusion Matrix: \n',confusion_matrix(y_test,rf_pred))
print('LogisticRegression Confusion Matrix: \n',confusion_matrix(y_test,lr_pred))
print('DecisionTreeClassifier Confusion Matrix: \n',confusion_matrix(y_test,dt_pred))

RandomForestClassifier Confusion Matrix: 
 [[2973  874]
 [ 110   43]]
LogisticRegression Confusion Matrix: 
 [[2523 1324]
 [  69   84]]
DecisionTreeClassifier Confusion Matrix: 
 [[2994  853]
 [ 109   44]]


In [177]:
# Classification Report
print('RandomForestClassifier Classification Report: \n',classification_report(y_test,rf_pred))
print('LogisticRegression Classification Report: \n',classification_report(y_test,lr_pred))
print('DecisionTreeClassifier Classification Report: \n',classification_report(y_test,dt_pred))


RandomForestClassifier Classification Report: 
               precision    recall  f1-score   support

           0       0.96      0.77      0.86      3847
           1       0.05      0.28      0.08       153

    accuracy                           0.75      4000
   macro avg       0.51      0.53      0.47      4000
weighted avg       0.93      0.75      0.83      4000

LogisticRegression Classification Report: 
               precision    recall  f1-score   support

           0       0.97      0.66      0.78      3847
           1       0.06      0.55      0.11       153

    accuracy                           0.65      4000
   macro avg       0.52      0.60      0.45      4000
weighted avg       0.94      0.65      0.76      4000

DecisionTreeClassifier Classification Report: 
               precision    recall  f1-score   support

           0       0.96      0.78      0.86      3847
           1       0.05      0.29      0.08       153

    accuracy                           0.7

In [123]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [124]:
from xgboost import XGBClassifier
xgb=XGBClassifier(n_estimators=500, learning_rate=0.05, eval_metric='aucpr',random_state=42, scale_pos_weight=10)
xgb.fit(x_train_smote,y_train_smote)
xgb_pred = xgb.predict(x_test_sf)
xgb_pred_proba = xgb.predict_proba(x_test_sf)[:, 1]

In [125]:
# Accuracy Score
print('XGBClassifier Accuracy Score: ',accuracy_score(y_test,xgb_pred))
# ROC AUC Score
print('XGBClassifier ROC AUC Score: ',roc_auc_score(y_test,xgb_pred_proba))
# Confusion Matrix
print('XGBClassifier Confusion Matrix: \n',confusion_matrix(y_test,xgb_pred))
# Classification Report
print('XGBClassifier Classification Report: \n',classification_report(y_test,xgb_pred))

XGBClassifier Accuracy Score:  0.62875
XGBClassifier ROC AUC Score:  0.5104724672990243
XGBClassifier Confusion Matrix: 
 [[2458 1389]
 [  96   57]]
XGBClassifier Classification Report: 
               precision    recall  f1-score   support

           0       0.96      0.64      0.77      3847
           1       0.04      0.37      0.07       153

    accuracy                           0.63      4000
   macro avg       0.50      0.51      0.42      4000
weighted avg       0.93      0.63      0.74      4000



In [129]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [133]:
params ={'max_depth':[3,5,6,7,8,10], 'min_samples_split':[20,40,60,80,100,120]}
dt1= DecisionTreeClassifier()
grid_search= GridSearchCV(estimator=dt, param_grid=params, cv=5)
grid_search.fit(x_train_smote, y_train_smote)


GridSearchCV(cv=5,
             estimator=DecisionTreeClassifier(class_weight='balanced',
                                              random_state=42),
             param_grid={'max_depth': [3, 5, 6, 7, 8, 10],
                         'min_samples_split': [20, 40, 60, 80, 100, 120]})

In [135]:
print(grid_search.best_params_)

{'max_depth': 10, 'min_samples_split': 20}


In [149]:
params ={'max_depth':[3,5,6,7,8,10], 'min_samples_split':[20,40,60,80,100,120]}
dt1= DecisionTreeClassifier()
rand_srch= RandomizedSearchCV(estimator=dt1, param_distributions=params, cv=5)
rand_srch.fit(x_train_smote, y_train_smote)


RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(),
                   param_distributions={'max_depth': [3, 5, 6, 7, 8, 10],
                                        'min_samples_split': [20, 40, 60, 80,
                                                              100, 120]})

In [151]:
print(rand_srch.best_params_)

{'min_samples_split': 20, 'max_depth': 10}


In [161]:
params ={'max_depth':[3,5,6,7,8,10], 'min_samples_split':[20,40,60,80,100,120]}
rf1= RandomForestClassifier()
rf_grid_search= GridSearchCV(estimator=dt, param_grid=params, cv=5)
rf_grid_search.fit(x_train_smote, y_train_smote)

GridSearchCV(cv=5,
             estimator=DecisionTreeClassifier(class_weight='balanced',
                                              max_depth=10,
                                              min_samples_split=20,
                                              random_state=42),
             param_grid={'max_depth': [3, 5, 6, 7, 8, 10],
                         'min_samples_split': [20, 40, 60, 80, 100, 120]})

In [159]:
params ={'max_depth':[3,5,6,7,8,10], 'min_samples_split':[20,40,60,80,100,120]}
rf1= RandomForestClassifier()
rf_rand_srch= RandomizedSearchCV(estimator=dt1, param_distributions=params, cv=5)
rf_rand_srch.fit(x_train_smote, y_train_smote)


RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(),
                   param_distributions={'max_depth': [3, 5, 6, 7, 8, 10],
                                        'min_samples_split': [20, 40, 60, 80,
                                                              100, 120]})

In [163]:
print(rf_grid_search.best_params_)

{'max_depth': 10, 'min_samples_split': 20}


In [165]:
print(rf_rand_srch.best_params_)

{'min_samples_split': 120, 'max_depth': 10}
